# GPU Summarization Server v2

Hosts **every** summarization model for the experiment on one Colab T4, so the
laptop only runs TextRank and the metric computation.

**Models served**

| key | model | context | dtype |
|---|---|---|---|
| `distilbart` | sshleifer/distilbart-cnn-12-6 | 1024 | fp16 |
| `bart-large` | facebook/bart-large-cnn | 1024 | fp16 |
| `pegasus` | google/pegasus-cnn_dailymail | 1024 | fp16 |
| `mt5-xlsum` | csebuetnlp/mT5_multilingual_XLSum | 512 | **fp32** |
| `led-large-book` | pszemraj/led-large-book-summary | 16384 | fp16 |
| `flan-t5-large` | google/flan-t5-large | 1024 | **fp32** |
| `turkish-mbart` | mukayese/mbart-large-turkish-summarization | 1024 | fp16 |
| `qwen-llm` | Qwen/Qwen2.5-7B-Instruct (4-bit) | 32k | 4-bit |

**Two things that matter**

1. **T5-family models stay in fp32.** mT5 and Flan-T5 overflow in fp16 and emit
   NaNs or garbage; bf16 would fix it but the T4 does not support bf16. Silent
   garbage would poison the experiment, so they are loaded in fp32.
2. **Models are loaded on demand and evicted under a VRAM budget**, rather than
   all held resident. A model switch costs a few seconds; running out of VRAM
   mid-experiment costs the whole run.

**First time:** Runtime → Change runtime type → **T4 GPU**, then Run all.
Paste the printed URL into your local `.env` as `COLAB_GPU_URL=...`.

---

## Restarting tomorrow

A new Colab session is a brand-new machine: installed packages, downloaded
model weights and the Python process are all gone. Nothing carries over.

**Run cells 1 → 8, then cell 10.** Skip cell 9 (the self-test) unless
something seems broken.

Cell 8 pre-downloads every model's weights before the server takes traffic.
Do not skip it: if a model downloads *during* a request instead, the caller
waits minutes and the tunnel returns an error page rather than JSON.

Then, on the laptop:

1. Copy the new URL printed by cell 7 — **it is different every session**.
2. Put it in `.env` as `COLAB_GPU_URL=...`, replacing the old line.
3. Restart the local server (`.env` is only read at startup).
4. Confirm with `curl localhost:8000/colab/status`.

Model weights re-download each session (~20 GB in total across all eight), but
loading is lazy, so only the models you actually use are fetched.


In [ ]:
# Cell 1: dependencies
!pip install -q flask pyngrok transformers torch sentencepiece protobuf accelerate bitsandbytes
print("dependencies installed")

In [ ]:
# Cell 2: GPU check
import torch

assert torch.cuda.is_available(), "NO GPU! Runtime > Change runtime type > T4 GPU"
props = torch.cuda.get_device_properties(0)
# NOTE: the attribute is total_memory. The v1 notebook used total_mem, which
# does not exist and made this cell raise before printing anything.
print(f"GPU:  {props.name}")
print(f"VRAM: {props.total_memory / 1e9:.1f} GB")
print(f"bf16 supported: {torch.cuda.is_bf16_supported()}  (False on T4 -> T5 models must use fp32)")

In [ ]:
# Cell 3: model registry and the lazy loader
#
# Each model declares the VRAM its weights need, so the loader can evict by a
# byte budget instead of a model count: three small models are fine, three
# large ones are not.

import gc, time
from collections import OrderedDict
import torch

# Budget for *weights only*. The rest of the 15 GB usable on a T4 is for
# activations, and LED at 16k tokens with beam search is the hungry case.
#
# All eight models resident at once would be 15.2 GB of weights — more than the
# card has before a single activation is allocated. Hence: load on demand,
# evict least-recently-used, and keep well under the line.
VRAM_BUDGET_GB = 7.5

MODELS = {
    "distilbart": {
        "repo": "sshleifer/distilbart-cnn-12-6", "family": "bart",
        "dtype": torch.float16, "max_input": 1024, "vram_gb": 0.7,
    },
    "bart-large": {
        "repo": "facebook/bart-large-cnn", "family": "bart",
        "dtype": torch.float16, "max_input": 1024, "vram_gb": 0.9,
    },
    "pegasus": {
        "repo": "google/pegasus-cnn_dailymail", "family": "pegasus",
        "dtype": torch.float16, "max_input": 1024, "vram_gb": 1.2,
    },
    "mt5-xlsum": {
        # fp32: mT5 overflows in fp16 on a T4 and returns NaNs.
        "repo": "csebuetnlp/mT5_multilingual_XLSum", "family": "mt5",
        "dtype": torch.float32, "max_input": 512, "vram_gb": 2.4,
    },
    "led-large-book": {
        "repo": "pszemraj/led-large-book-summary", "family": "led",
        "dtype": torch.float16, "max_input": 16384, "vram_gb": 1.0,
    },
    "flan-t5-large": {
        # fp32 for the same reason as mT5.
        "repo": "google/flan-t5-large", "family": "flan",
        "dtype": torch.float32, "max_input": 1024, "vram_gb": 3.2,
    },
    "turkish-mbart": {
        "repo": "mukayese/mbart-large-turkish-summarization", "family": "mbart",
        "dtype": torch.float16, "max_input": 1024, "vram_gb": 1.3,
    },
    "qwen-llm": {
        # Instruction-tuned LLM datapoint that costs nothing to run.
        #
        # Pre-quantized 4-bit weights: 5.5 GB to download per session. The
        # original Qwen/Qwen2.5-7B-Instruct would pull 15.2 GB and only then
        # quantize, and Colab re-downloads everything on every restart.
        # max_input reduced from 8192 to 3072, and vram_gb corrected from 5.5
        # to the MEASURED 9.3 GB. The 5.5 GB figure covers only the quantized
        # transformer weights; the embedding and output layers stay in fp16
        # (152k vocab x 3584 dims, twice) and the loader needs working memory.
        # At 8192 tokens the KV cache then had nowhere to live and generation
        # OOM'd on a 14.56 GB T4. 3072 tokens means more chunking but it runs.
        "repo": "unsloth/Qwen2.5-7B-Instruct-bnb-4bit", "family": "llm",
        "dtype": "4bit", "max_input": 3072, "vram_gb": 9.3,
    },
}

ENABLE_LLM = True          # set False to drop qwen-llm entirely
if not ENABLE_LLM:
    MODELS.pop("qwen-llm", None)


def _from_pretrained(cls, repo, **kwargs):
    """from_pretrained that degrades gracefully if safetensors are unavailable."""
    try:
        return cls.from_pretrained(repo, **kwargs)
    except Exception:
        kwargs.pop("use_safetensors", None)
        return cls.from_pretrained(repo, **kwargs)


class ModelManager:
    """Loads models on demand and evicts the least recently used under a budget."""

    def __init__(self, budget_gb=VRAM_BUDGET_GB):
        self.budget_gb = budget_gb
        self.loaded = OrderedDict()      # key -> {"model","tokenizer","vram_gb"}
        self.load_times = {}

    def _resident_gb(self):
        return sum(entry["vram_gb"] for entry in self.loaded.values())

    def _evict_until(self, needed_gb):
        while self.loaded and self._resident_gb() + needed_gb > self.budget_gb:
            key, entry = self.loaded.popitem(last=False)     # least recently used
            print(f"  evicting {key} ({entry['vram_gb']} GB)")
            entry["model"] = None
            entry["tokenizer"] = None
            del entry
            gc.collect()
            torch.cuda.empty_cache()
        # Trust the GPU, not the bookkeeping. A caught exception's traceback can
        # keep an evicted model's tensors alive, so memory_allocated may far
        # exceed what we think is resident -- which is how a load OOMs with
        # "enough" budget left.
        actually_held = torch.cuda.memory_allocated() / 1e9
        if actually_held > self._resident_gb() + 1.0:
            print(f"    {actually_held:.1f} GB still held vs {self._resident_gb():.1f} GB tracked; "
                  f"clearing leaked references")
            import sys
            if hasattr(sys, "last_traceback"):
                sys.last_traceback = None
            sys.exc_info()
            for _ in range(3):
                gc.collect()
                torch.cuda.empty_cache()
            print(f"    after cleanup: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

    def get(self, key):
        if key not in MODELS:
            raise KeyError(f"unknown model '{key}'. Available: {list(MODELS)}")
        if key in self.loaded:
            self.loaded.move_to_end(key)                     # mark as recently used
            return self.loaded[key]["model"], self.loaded[key]["tokenizer"]

        spec = MODELS[key]
        self._evict_until(spec["vram_gb"])
        started = time.time()
        print(f"  loading {key} ({spec['repo']}, {spec['dtype']}) ...")
        before = torch.cuda.memory_allocated()
        try:
            model, tokenizer = self._load(key, spec)
        except torch.cuda.OutOfMemoryError:
            # The OOM guard used to wrap only generation, which is the one place
            # it was never needed: a 5 GB model fails at load time.
            print(f"    OOM loading {key}; dropping everything and retrying once")
            self.drop_all()
            before = torch.cuda.memory_allocated()
            model, tokenizer = self._load(key, spec)
        # Budget on the MEASURED footprint, not the hand-written estimate: a
        # wrong estimate is how an eviction policy quietly overcommits a GPU.
        measured_gb = round(max(torch.cuda.memory_allocated() - before, 0) / 1e9, 2)
        if measured_gb > 0:
            if measured_gb > spec["vram_gb"] * 1.25:
                print(f"    note: {key} measured {measured_gb} GB vs {spec['vram_gb']} GB estimated")
            spec["vram_gb"] = measured_gb          # future evictions use the truth
        # transformers renamed torch_dtype -> dtype and warns about it. Rather
        # than guess which name this version wants, check what actually loaded
        # and correct it: a model silently sitting in fp32 would blow the VRAM
        # budget, and one silently in fp16 would emit NaNs for T5.
        if spec["dtype"] in (torch.float16, torch.float32):
            actual = next(model.parameters()).dtype
            if actual != spec["dtype"]:
                print(f"    correcting dtype: loaded as {actual}, wanted {spec['dtype']}")
                model = model.to(spec["dtype"])
        self.loaded[key] = {"model": model, "tokenizer": tokenizer,
                            "vram_gb": measured_gb or spec["vram_gb"]}
        self.load_times[key] = round(time.time() - started, 1)
        print(f"  loaded {key} in {self.load_times[key]}s; "
              f"resident {self._resident_gb():.1f}/{self.budget_gb} GB; "
              f"allocated {torch.cuda.memory_allocated()/1e9:.1f} GB")
        return model, tokenizer

    def _load(self, key, spec):
        from transformers import (AutoModelForCausalLM, AutoModelForSeq2SeqLM, AutoTokenizer,
                                  LEDForConditionalGeneration, LEDTokenizer,
                                  MBart50TokenizerFast, MBartForConditionalGeneration)
        repo, family = spec["repo"], spec["family"]
        # Several repos ship both pytorch_model.bin and model.safetensors, and
        # the loader downloads whichever it finds first -- sometimes both,
        # doubling the disk used. Ask for safetensors, fall back if a repo
        # only has the older format.
        extra = {"use_safetensors": True}

        if family == "led":
            tokenizer = LEDTokenizer.from_pretrained(repo)
            model = _from_pretrained(LEDForConditionalGeneration, repo,
                                     torch_dtype=spec["dtype"], **extra).to("cuda").eval()
        elif family == "mbart":
            tokenizer = MBart50TokenizerFast.from_pretrained(repo)
            model = _from_pretrained(MBartForConditionalGeneration, repo,
                                     torch_dtype=spec["dtype"], **extra).to("cuda").eval()
        elif family == "llm":
            tokenizer = AutoTokenizer.from_pretrained(repo)
            if "4bit" in repo or "bnb" in repo:
                # Already quantized: the config travels with the weights, and
                # passing our own quantization_config would conflict with it.
                model = AutoModelForCausalLM.from_pretrained(repo, device_map="cuda").eval()
            else:
                from transformers import BitsAndBytesConfig
                quant = BitsAndBytesConfig(load_in_4bit=True,
                                           bnb_4bit_compute_dtype=torch.float16,
                                           bnb_4bit_quant_type="nf4")
                model = AutoModelForCausalLM.from_pretrained(
                    repo, quantization_config=quant, device_map="cuda").eval()
        else:
            tokenizer = AutoTokenizer.from_pretrained(repo)
            model = _from_pretrained(AutoModelForSeq2SeqLM, repo,
                                     torch_dtype=spec["dtype"], **extra).to("cuda").eval()
        return model, tokenizer

    def drop_all(self):
        """Release every resident model (the out-of-memory escape hatch)."""
        for key in list(self.loaded):
            entry = self.loaded.pop(key)
            del entry["model"], entry["tokenizer"], entry
        gc.collect()
        torch.cuda.empty_cache()
        print(f"    dropped all models; allocated now {torch.cuda.memory_allocated()/1e9:.2f} GB")

    def status(self):
        return {
            "resident": list(self.loaded.keys()),
            "resident_gb": round(self._resident_gb(), 2),
            "budget_gb": self.budget_gb,
            "allocated_gb": round(torch.cuda.memory_allocated() / 1e9, 2),
            "load_times": self.load_times,
        }


manager = ModelManager()

_total = sum(spec["vram_gb"] for spec in MODELS.values())
_usable = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"registry ready: {list(MODELS)}")
print(f"  all models resident would need ~{_total:.1f} GB of weights; the card has "
      f"~{_usable:.1f} GB total")
print(f"  budget {VRAM_BUDGET_GB} GB of weights resident, leaving ~{_usable - VRAM_BUDGET_GB:.1f} GB "
      f"for activations")
print("  models load on demand and the least recently used is evicted — a switch costs seconds, "
      "an OOM costs the run")

In [ ]:
# Cell 4: summarization with hierarchical chunking
#
# Mirrors the local pipeline: split -> summarize each chunk -> merge ->
# summarize the merge, recursing while the merge is still too long. Chunking
# happens here (not on the laptop) so the whole document can be sent as-is.

import re
import torch

WHITESPACE = lambda s: re.sub(r"\s+", " ", re.sub(r"\n+", " ", s.strip()))


def split_into_chunks(text, tokenizer, max_tokens, overlap_sentences=2):
    """Sentence-aligned chunks that fit the model's context."""
    sentences = [s for s in re.split(r"(?<=[.!?])\s+", text.strip()) if s.strip()]
    chunks, current, length = [], [], 0
    for sentence in sentences:
        n = len(tokenizer.encode(sentence, add_special_tokens=False))
        if length + n > max_tokens and current:
            chunks.append(" ".join(current))
            current = current[-overlap_sentences:] if overlap_sentences else []
            length = sum(len(tokenizer.encode(s, add_special_tokens=False)) for s in current)
        current.append(sentence)
        length += n
    if current:
        chunks.append(" ".join(current))
    return chunks or [text]


def generate_one(key, text, max_length, min_length):
    """
    One generation pass, with an out-of-memory safety net.

    If the GPU runs out of memory, every resident model is dropped and the
    generation is retried greedily (no beams). Beam search is the first thing
    to sacrifice: a greedy summary is worth far more than a dead run.
    """
    try:
        return _generate(key, text, max_length, min_length)
    except torch.cuda.OutOfMemoryError:
        print(f"    OOM on {key}: dropping resident models and retrying greedily")
        manager.drop_all()
        return _generate(key, text, max_length, min_length, force_greedy=True)


@torch.no_grad()
def _generate(key, text, max_length, min_length, force_greedy=False):
    """One generation pass for one model on text that already fits its context."""
    spec = MODELS[key]
    model, tokenizer = manager.get(key)
    family = spec["family"]
    min_length = max(10, min(min_length, max_length - 10))

    if family == "llm":
        target = max(40, int(max_length * 0.75))
        messages = [
            {"role": "system", "content":
             "You are a professional text summarizer. Summarize accurately and never invent "
             "facts that are not in the source. Reply in the same language as the input."},
            {"role": "user", "content":
             f"Summarize the following text in approximately {target} words.\n\nTEXT:\n{text}"},
        ]
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True,
                           max_length=spec["max_input"]).to("cuda")
        output = model.generate(**inputs, max_new_tokens=max_length, do_sample=False,
                                repetition_penalty=1.05,
                                pad_token_id=tokenizer.eos_token_id)
        generated = output[0][inputs["input_ids"].shape[1]:]
        return tokenizer.decode(generated, skip_special_tokens=True).strip()

    if family == "led":
        # Dynamic padding: v1 padded every request to 16384 tokens, so a two-page
        # document cost the same as a book chapter.
        inputs = tokenizer(text, return_tensors="pt", max_length=spec["max_input"],
                           truncation=True).to("cuda")
        input_tokens = inputs["input_ids"].shape[1]
        global_attention_mask = torch.zeros_like(inputs["input_ids"])
        global_attention_mask[:, 0] = 1
        # Cross-attention keys/values are held per beam over the whole encoder
        # output, so beams x 16k tokens is where LED blows up. Back off on long
        # inputs rather than risking an OOM.
        beams = 1 if force_greedy else (4 if input_tokens <= 4096 else (2 if input_tokens <= 10000 else 1))
        output = model.generate(**inputs, global_attention_mask=global_attention_mask,
                                max_length=max_length, min_length=min_length,
                                num_beams=beams, length_penalty=2.0, no_repeat_ngram_size=3)
        return tokenizer.decode(output[0], skip_special_tokens=True).strip()

    if family == "mbart":
        tokenizer.src_lang = "tr_TR"
        inputs = tokenizer(text, return_tensors="pt", max_length=spec["max_input"],
                           truncation=True).to("cuda")
        # lang_code_to_id was removed from newer transformers; this is the
        # supported way to get the forced BOS token.
        forced_bos = tokenizer.convert_tokens_to_ids("tr_TR")
        output = model.generate(**inputs, max_length=max_length, min_length=min_length,
                                num_beams=1 if force_greedy else 4, length_penalty=1.0,
                                no_repeat_ngram_size=3, forced_bos_token_id=forced_bos)
        return tokenizer.decode(output[0], skip_special_tokens=True).strip()

    if family == "flan":
        prompt = ("Summarize the following text concisely, keeping only the most important "
                  f"facts and details:\n\n{text}")
        inputs = tokenizer(prompt, return_tensors="pt", max_length=spec["max_input"],
                           truncation=True).to("cuda")
        output = model.generate(**inputs, max_length=max_length, min_length=min_length,
                                num_beams=1 if force_greedy else 4, length_penalty=1.0,
                                no_repeat_ngram_size=3, repetition_penalty=2.5)
        return tokenizer.decode(output[0], skip_special_tokens=True).strip()

    # bart / pegasus / mt5
    #
    # Pegasus/CNN-DailyMail writes "<n>" where it means a line break. Left in,
    # it lands in the summary text and is counted as a token by every metric.
    payload = WHITESPACE(text) if family == "mt5" else text
    inputs = tokenizer(payload, return_tensors="pt", max_length=spec["max_input"],
                       truncation=True).to("cuda")
    output = model.generate(**inputs, max_length=max_length, min_length=min_length,
                            num_beams=1 if force_greedy else 4, length_penalty=2.0,
                            no_repeat_ngram_size=3)
    decoded = tokenizer.decode(output[0], skip_special_tokens=True).strip()
    if family == "pegasus":
        decoded = re.sub(r"\s*<n>\s*", " ", decoded).strip()
    return decoded


def summarize(key, text, max_length=300, min_length=60):
    """Hierarchical summarization: chunk, summarize each, merge, summarize again."""
    spec = MODELS[key]
    _, tokenizer = manager.get(key)
    # Leave room for the prompt and the output.
    chunk_limit = int(spec["max_input"] * 0.85)
    token_count = len(tokenizer.encode(text, add_special_tokens=False))

    if token_count <= chunk_limit:
        return generate_one(key, text, max_length, min_length), {"chunks": 1, "passes": 1}

    chunks = split_into_chunks(text, tokenizer, chunk_limit)
    chunk_max = max(80, max_length // max(1, len(chunks) // 2))
    chunk_min = max(20, chunk_max // 3)
    print(f"    [{key}] {token_count} tokens -> {len(chunks)} chunks")
    summaries = [generate_one(key, chunk, chunk_max, chunk_min) for chunk in chunks]

    merged = " ".join(summaries)
    passes = 2
    while len(tokenizer.encode(merged, add_special_tokens=False)) > chunk_limit and passes < 5:
        sub_chunks = split_into_chunks(merged, tokenizer, chunk_limit)
        merged = " ".join(generate_one(key, c, chunk_max, chunk_min) for c in sub_chunks)
        passes += 1

    final = generate_one(key, merged, max_length, min_length)
    return final, {"chunks": len(chunks), "passes": passes}


print("summarization ready")

In [ ]:
# Cell 5: Flask API
#
# /health    what the server is and which models it can serve
# /models    the registry with context sizes and dtypes
# /summarize {text, model, max_length, min_length} -> summary + timing
# /status    what is currently resident in VRAM

import time, traceback
from flask import Flask, jsonify, request

app = Flask(__name__)
REQUEST_LOG = []


@app.route("/health")
def health():
    return jsonify({
        "status": "running",
        "gpu": torch.cuda.get_device_name(0),
        "vram_total_gb": round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1),
        "models": list(MODELS.keys()),
        "resident": manager.status(),
        "version": 2,
    })


@app.route("/models")
def models():
    return jsonify({key: {"repo": spec["repo"], "max_input": spec["max_input"],
                          "dtype": str(spec["dtype"]), "vram_gb": spec["vram_gb"]}
                    for key, spec in MODELS.items()})


@app.route("/status")
def status():
    return jsonify({"resident": manager.status(), "recent_requests": REQUEST_LOG[-20:]})


@app.route("/summarize", methods=["POST"])
def summarize_endpoint():
    data = request.get_json(force=True)
    text = (data.get("text") or "").strip()
    key = data.get("model", "led-large-book")
    if not text:
        return jsonify({"error": "no text provided"}), 400
    if key not in MODELS:
        return jsonify({"error": f"unknown model '{key}'", "available": list(MODELS)}), 400

    started = time.time()
    try:
        summary, info = summarize(key, text,
                                  max_length=int(data.get("max_length", 300)),
                                  min_length=int(data.get("min_length", 60)))
        elapsed = time.time() - started
        record = {"model": key, "input_words": len(text.split()),
                  "summary_words": len(summary.split()),
                  "gpu_seconds": round(elapsed, 2), **info}
        REQUEST_LOG.append(record)
        # gpu_seconds is returned so the local cost accounting can price GPU
        # time separately from network round-trip time.
        return jsonify({"summary": summary, "model": key, **record})
    except Exception as error:
        return jsonify({"error": str(error), "traceback": traceback.format_exc()}), 500


print("Flask app ready")

In [ ]:
# Cell 6: your ngrok token
# Free token: https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_TOKEN = ""   # <-- paste your token here

In [ ]:
# Cell 7: start the server and open the tunnel  (safe to re-run)
import socket
import threading
import time

import requests
from pyngrok import ngrok

assert NGROK_TOKEN, "Paste your ngrok token in the cell above first"


def port_in_use(port=5000):
    with socket.socket() as probe:
        return probe.connect_ex(("127.0.0.1", port)) == 0


# Re-running this cell used to try to start a second Flask server and fail with
# "Address already in use", which looks alarming but is harmless: the first
# server is still serving. Now it simply reuses the running one.
if port_in_use():
    print("Flask is already serving on port 5000 - reusing it")
else:
    threading.Thread(target=lambda: app.run(host="0.0.0.0", port=5000, threaded=True),
                     daemon=True).start()
    for _ in range(20):
        if port_in_use():
            break
        time.sleep(0.5)
    print("Flask started on port 5000")

# Reconnect the tunnel. This is the part that actually goes down when Colab
# drops the connection, and re-running this cell is the fix.
ngrok.kill()
ngrok.set_auth_token(NGROK_TOKEN)
public_url = ngrok.connect(5000).public_url

# Verify end to end before claiming success: the keep-alive cell prints the URL
# from a variable and will happily keep printing it after the tunnel dies.
reachable = False
for attempt in range(5):
    try:
        probe = requests.get(f"{public_url}/health",
                             headers={"ngrok-skip-browser-warning": "true"}, timeout=20)
        if probe.status_code == 200 and "models" in probe.text:
            reachable = True
            break
    except Exception:
        pass
    time.sleep(2)

print("=" * 66)
if reachable:
    print(f"SERVER RUNNING AND REACHABLE: {public_url}")
else:
    print(f"TUNNEL NOT ANSWERING: {public_url}")
    print("Re-run this cell. If it keeps failing, restart the runtime and run cells 1-8.")
print("=" * 66)
print("")
print("Put this line in your local .env (it is unchanged if your domain is static):")
print(f"COLAB_GPU_URL={public_url}")
print("")
print(f"Test from the laptop:  curl -s {public_url}/health")


In [ ]:
# Cell 8: pre-download all model weights  (RUN THIS BEFORE THE EXPERIMENT)
#
# Weights are fetched from HuggingFace the first time a model is used. If that
# happens inside an HTTP request, the caller waits minutes for a download and
# the tunnel usually gives up and returns an error page instead of JSON --
# which is exactly what broke the first self-test on mt5-xlsum.
#
# This loads each model once, in-process, then releases them. The files stay in
# the local HF cache for the rest of the session, so every later load is from
# disk (seconds) and no request ever waits on a download.

import time

start = time.time()
for key in MODELS:
    try:
        t0 = time.time()
        manager.get(key)          # downloads if needed, then loads
        print(f"  ready: {key:16} ({time.time() - t0:.0f}s)")
    except Exception as error:
        print(f"  FAILED: {key:16} {type(error).__name__}: {str(error)[:160]}")
    manager.drop_all()            # keep VRAM free; the download is what we wanted

print(f"\nall weights cached in {(time.time() - start) / 60:.1f} min")
print("later loads come from local disk, so requests only pay generation time")

In [ ]:
# Cell 9: self-test (OPTIONAL — skip it on a daily restart)
#
# QUICK_TEST=True  checks two models, ~1 minute.
# QUICK_TEST=False checks all eight: every model downloads and loads, which on
#                  a fresh Colab VM means ~20 GB of downloads and 10-15 minutes.
# Models download lazily anyway, so skipping this just defers the cost to the
# first real request for each model.
QUICK_TEST = True

import requests, time

time.sleep(3)
print("health:", requests.get(f"{public_url}/health").json()["models"])

en = ("Artificial intelligence has moved from a theoretical idea to one of the most "
      "consequential technologies of the century. Modern systems rest on neural networks "
      "trained by backpropagation on very large datasets. Large language models now write "
      "essays, generate code and translate between languages. In healthcare, models assist "
      "with disease prediction and drug discovery, while in transport they underpin driver "
      "assistance. The same capabilities raise questions about employment, privacy and the "
      "concentration of power in a few firms, which regulators are only beginning to address.")

tr = ("Yapay zeka, bilgisayar bilimlerinde teorik bir kavramdan günümüzün en önemli "
      "teknolojilerinden birine dönüştü. Modern yapay zekanın temeli, çok büyük veri "
      "kümeleri üzerinde eğitilen yapay sinir ağlarına dayanmaktadır. Büyük dil modelleri "
      "metin üretme ve çeviri konusunda olağanüstü yetenekler göstermiştir. Sağlık "
      "sektöründe hastalıkların erken teşhisinde kullanılmakta, ulaşımda ise sürücü destek "
      "sistemlerinin temelini oluşturmaktadır. Bu gelişmeler istihdam ve mahremiyet "
      "konusunda önemli soruları da beraberinde getirmektedir.")

test_keys = ["distilbart", "turkish-mbart"] if QUICK_TEST else list(MODELS)
for key in test_keys:
    text = tr if key == "turkish-mbart" else en
    raw = requests.post(f"{public_url}/summarize",
                        json={"text": text, "model": key,
                              "max_length": 90, "min_length": 30},
                        timeout=900)
    try:
        response = raw.json()
    except Exception:
        # Usually the tunnel's own error page: the request outlived the edge
        # timeout, typically because the model was still downloading.
        print(f"\n[{key}] non-JSON reply (HTTP {raw.status_code}): {raw.text[:200]}")
        continue
    if "error" in response:
        print(f"\n[{key}] ERROR: {response['error'][:160]}")
    else:
        print(f"\n[{key}] {response['gpu_seconds']}s, {response['summary_words']}w")
        print("   ", response["summary"][:220])

In [ ]:
# Cell 10: keep alive (stop this cell to shut the server down)
import time

print(f"Server: {public_url}")
print("Leave this running for the duration of the experiment.")
while True:
    time.sleep(120)
    print(".", end="", flush=True)

In [ ]:
# Cell 11: disk usage (optional)
#
# Colab disk does not shrink on its own: downloaded weights live in the
# HuggingFace cache for the life of the VM. That is deliberate -- it is what
# makes a model reload take seconds instead of minutes.
#
# To reset the disk completely: Runtime > Disconnect and delete runtime.
# That gives a brand-new VM, and the next run re-downloads everything.

import subprocess

print(subprocess.run(["df", "-h", "/"], capture_output=True, text=True).stdout)
print("HuggingFace cache, largest first:")
print(subprocess.run(
    "du -sh ~/.cache/huggingface/hub/models--* 2>/dev/null | sort -rh",
    shell=True, capture_output=True, text=True).stdout or "  (empty)")

# Free one model you are finished with (it re-downloads if needed later):
#   !rm -rf ~/.cache/huggingface/hub/models--google--pegasus-cnn_dailymail
# Free everything cached:
#   !rm -rf ~/.cache/huggingface/hub/models--*
